[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pctablet505/litert-demo/blob/main/litertlm_export_demo.ipynb)

# LiteRT-LM Export Demo — Gemma 3 270M

Minimal demonstration of exporting a KerasHub `Gemma3CausalLM` to a **LiteRT-LM bundle** (`.litertlm`).

LiteRT-LM packages the TFLite model (with `prefill` + `decode` signatures), the SentencePiece tokenizer, and LLM metadata into a single file. On Android, the LiteRT-LM runtime handles tokenization, KV-cache management, sampling, and streaming for you.

> **Requirements:**
> - `KERAS_BACKEND=torch`
> - `litert-torch` and `litert-lm-builder` installed
> - This export path is **PyTorch-only**
> - LiteRT-LM PR branch: `torch-backend-litert-minimal-litertlm`

In [6]:
!pip install -q litert-torch
!pip install -q git+https://github.com/keras-team/keras.git
!pip install -q git+https://github.com/pctablet505/keras-hub.git@torch-backend-litert-minimal-litertlm
!pip install --upgrade protobuf




  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 19.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.153.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.0 which is incompatible.
ydf 0.15.0 requires protobuf<7.

In [4]:
import os

PRESET = "hf://google/gemma-3-270m-it"

CACHE_LEN = 1024   # Max conversation context
PREFILL_LEN = 128  # Max prompt length per turn

print("Preset:", PRESET)
print(f"cache_length={CACHE_LEN}, prefill_seq_len={PREFILL_LEN}")

Preset: hf://google/gemma-3-270m-it
cache_length=1024, prefill_seq_len=128


In [5]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import keras_hub

model = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)
model.preprocessor.sequence_length = CACHE_LEN

# Build via direct call with concrete shapes
processed = model.preprocessor({"prompts": ["What is Keras?"], "responses": [""]})
model(processed[0])
print("Model built. Output shape:", model(processed[0]).shape)

Model built. Output shape: torch.Size([1, 1024, 262144])


## Export to `.litertlm`

A single call with `format="litertlm"` produces the bundle. The exporter automatically:
- Traces `prefill` (prompt → KV cache) and `decode` (single token → logits + updated KV cache) signatures
- Bundles the SentencePiece tokenizer
- Writes `LlmMetadata` (start/stop tokens, context window, model type)

In [6]:
# Single bucket (legacy behavior)
# model.export("gemma3_270m_it.litertlm", format="litertlm", prefill_seq_len=128)

# Multiple buckets — runtime dispatches to the smallest fitting signature
# NOTE: Each bucket adds ~2-4 min to export time (signature is re-traced),
# but model size increase is minimal (~0.07% for 3 extra signatures)
# because weights are shared across all prefill signatures.
model.export(
    "gemma3_270m_it.litertlm",
    format="litertlm",
    prefill_seq_len=[32, 64],
)
print("✅ Exported with bucketing: prefill_32, prefill_64")
print("Size (MB):", round(os.path.getsize("gemma3_270m_it.litertlm") / 1e6, 1))

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: prefill_32

(00:24) [START] LiteRT-Torch Convert > Torch Export: prefill_32 > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(01:04) [ DONE] LiteRT-Torch Convert > Torch Export: prefill_32 > ExportedProgram Run Decompositions (+00:39)

...

(09:43) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:54)

(09:44) [ DONE] LiteRT-Torch Convert (+09:44)

(00:00) [START] Write Model to /tmp/tmps72xmejc/model.tflite

(00:03) [ DONE] Write Model to /tmp/tmps72xmejc/model.tflite (+00:03)

✅ Exported with bucketing: prefill_32, prefill_64
Size (MB): 1082.4


## Verify Bundle Contents

Peek inside the `.litertlm` file to confirm it contains the expected assets.

In [7]:
import io
from litert_lm_builder import litertlm_peek

output = io.StringIO()
litertlm_peek.peek_litertlm_file("gemma3_270m_it.litertlm", None, output)
print(output.getvalue())
print("✅ LiteRT-LM bundle verified.")

LiteRT-LM Version: 1.5.0

+------------------------------------------------+
|                System Metadata                 |
+------------------------------------------------+
  Key: Authors, Value (String): KerasHub
  Key: uuid, Value (String): 598be061-48aa-438e-a8f1-55b1fd9709dd
  Key: creation_timestamp, Value (String): 2026-06-05T05:17:07.413683+00:00

+------------------------------------------------+
|                  Sections (3)                  |
+------------------------------------------------+

Section 0:
  Items:
    Key: model_type, Value (String): tf_lite_prefill_decode
  Begin Offset: 16384
  End Offset:   1077720880
  Data Type:    TFLiteModel


Section 1:
  Items:
    <None>
  Begin Offset: 1077723136
  End Offset:   1082412210
  Data Type:    SP_Tokenizer


Section 2:
  Items:
    <None>
  Begin Offset: 1082425344
  End Offset:   1082425372
  Data Type:    LlmMetadataProto
  <<<<<<<< start of LlmMetadata
    start_token {
      token_ids {
        ids: 2
      }

### Bucketing Performance (Measured on Pixel 9)

| Model | Prompt Tokens | TTFT |
|-------|--------------|------|
| Fixed-128 (`prefill_seq_len=128`) | 31 tok | **2866.5 ms** |
| Bucketed `[32, 64, 128]` | 31 tok | **1626.4 ms** |

**→ 43.3% faster TTFT** with bucketing for a ~31-token prompt.

The fixed model pads every prompt to 128 tokens. The bucketed runtime dispatches to `prefill_64` (smallest fitting signature), avoiding ~60% of wasted prefill compute.

In [8]:
from litert_torch.generative.quantize.quant_recipes import full_dynamic_recipe

model.export(
    "gemma3_270m_it_wi8afp32.litertlm",
    format="litertlm",
    prefill_seq_len=[32, 64],
    quant_config=full_dynamic_recipe(),
)
print("✅ Exported + quantized in one step")
print("Size (MB):", round(os.path.getsize("gemma3_270m_it_wi8afp32.litertlm") / 1e6, 1))

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: prefill_32

(00:27) [START] LiteRT-Torch Convert > Torch Export: prefill_32 > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(01:06) [ DONE] LiteRT-Torch Convert > Torch Export: prefill_32 > ExportedProgram Run Decompositions (+00:39)

...

Applying Transformations to tensors:: 100%|██████████| 21775/21775 [00:00<00:00, 113243.19it/s]


Original model size: 1.00 GiB
Quantized model size: 273.86 MiB
Quantization Ratio: 0.27 (3.8x smaller)
Total time: 6.80 s


(10:10) [ DONE] LiteRT-Torch Convert > Quantize Model (+00:17)

(10:10) [ DONE] LiteRT-Torch Convert (+10:10)

✅ Exported + quantized in one step
Size (MB): 291.9


## Push to Android & Test

```bash
# Push to device
adb push gemma3_270m_it_wi8afp32.litertlm /data/local/tmp/
```

**Verified on Pixel 9 (physical ARM64):**
- Engine init: **2,318 ms** (vs ~5,336 ms for FP32)
- Generation: **4,213 ms** for "What is Keras?"
- Response: *"Keras is a high-level, Python API for building and training machine learning models..."*